# Chit 3 baseline: flagging RFI over the Karoo · **GPU-RT**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Score every pixel of a HERA waterfall as RFI or not, first with the **classical threshold flagger** (the baseline to beat), then with a tiny U-Net. Measure per-pixel AUPRC and F1 on the test waterfalls, and the **batch-1 latency on the T4, including the copies** to and from the GPU.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~10 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

> **Runtime → Change runtime type → T4 GPU** before running: the latency budget is defined on Colab's T4. (Without a GPU everything still runs, and the latency printed is the CPU's, labelled as such.)

In [ ]:
# --- Colab setup: run once per session (~1 min) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["onnx"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

In [ ]:
from common.timing import measure, cpu_call, gpu_call
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

## 0. Understand the data
Before any model, look at the data: what one sample is, what the classes look like, how the splits were made, and why the chit uses its metric.
This part only reads and plots (≲ 1 min). The model starts in Part 1.

### 0.1 The task in plain words
A radio telescope measures very faint radio waves from space. Human transmitters (FM radio, TV, aircraft, satellites) and lightning are far brighter, and they
spoil the data: this is **radio-frequency interference (RFI)**. Every pixel that contains RFI must be flagged (marked as bad) before the data are averaged and the raw data
are thrown away. For the next telescopes (SKA-Low), this must happen in real time on a GPU, one waterfall after another.
**One decision of the model** = for each pixel of a waterfall, a score: "this pixel contains RFI".

### 0.2 What one sample is
One sample is a **waterfall**: the signal of one HERA dish, or of one pair of dishes, as an image of **512 time steps × 512 frequency channels**
(shape `(1, 512, 512)`: one channel, then time, then frequency). Each pixel is an **amplitude |V|** (the strength of the measured signal, in the simulation's arbitrary units).
**Label:** a mask of the same shape, 1 = this pixel contains RFI, 0 = clean. The data are simulated, so the mask is **exact** (no human labelling errors).

In [ ]:
import matplotlib.pyplot as plt
from capstone.data.hera import load
X0, y0 = load("val")                              # 60 waterfalls: enough to look at
print("one waterfall:", X0.shape[1:], "= (channel, time step, frequency channel) | mask:", y0.shape[1:], y0.dtype)
print(f"amplitudes: {X0.min():.1e} to {X0.max():.0f} (all positive) | RFI pixels: {y0.mean():.2%}")

In [ ]:
rng = np.random.default_rng(0)
lx = np.log10(X0[:, 0]); m = y0[:, 0].astype(bool)
pick = lambda v: v[rng.choice(len(v), min(len(v), 300_000), replace=False)]
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
bins = np.linspace(-6, 3, 181)
ax[0].hist(pick(lx[~m]), bins, density=True, histtype="step", color="C0", label="clean pixels")
ax[0].hist(pick(lx[m]), bins, density=True, histtype="step", color="C3", label="RFI pixels")
ax[0].set(xlabel="log10 amplitude |V|", ylabel="density", title="Pixel values: RFI is brighter, but they overlap", yscale="log")
med = np.median(lx.reshape(len(lx), -1), 1)
ax[1].hist(med, 30, color="0.4")
ax[1].set(xlabel="median log10 amplitude of one waterfall", ylabel="waterfalls", title="Each waterfall has its own brightness")
ax[2].hist(100 * y0.reshape(len(y0), -1).mean(1), 30, color="C3")
ax[2].set(xlabel="RFI pixels in the waterfall [%]", ylabel="waterfalls", title="Fraction of flagged pixels per waterfall")
for a in ax[:1]:
    a.legend(fontsize=8)
plt.tight_layout(); plt.show()
z = lx - med[:, None, None]                       # each pixel relative to its own waterfall's median
print(f"faint waterfalls (median amplitude below 1): {np.mean(med < 0):.0%} of them; bright ones: {np.mean(med >= 0):.0%}")
print(f"RFI pixels brighter than their waterfall's median: {np.mean(z[m] > 0):.1%};  clean pixels brighter: {np.mean(z[~m] > 0):.1%}")

**What to notice.** Left: the amplitudes span about 8 orders of magnitude, so use a log scale. RFI pixels are brighter *on average*, but many clean pixels
are as bright as RFI pixels. The reason is in the middle plot: the waterfalls come in **two groups**, bright ones (median amplitude ~10–16) and faint ones (about a third), 100 to 500 times fainter.
A clean pixel of a bright waterfall has the same value as an RFI pixel of a faint one, so a single global threshold cannot work: judge each pixel against *its own* waterfall
(as the classical flagger does, row by row). Even then, about one RFI pixel in seven is fainter than its waterfall's median: weak RFI is hard to see.
Right: the share of RFI pixels varies from below 1% to about 6% per waterfall.

### 0.3 Look at it
Four validation waterfalls, from the least RFI (left) to the most (right). Top: log amplitude. Bottom: the exact RFI mask (black = RFI).

In [ ]:
frac = y0.reshape(len(y0), -1).mean(1)
show = np.argsort(frac)[[0, len(frac) // 3, 2 * len(frac) // 3, -1]]          # least RFI ... most RFI
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for j, i in enumerate(show):
    im = ax[0, j].imshow(np.log10(X0[i, 0]), aspect="auto", cmap="viridis", origin="lower")
    plt.colorbar(im, ax=ax[0, j], label="log10 |V|")
    ax[0, j].set(title=f"val waterfall {i}: log10 amplitude", xlabel="frequency channel", ylabel="time step")
    ax[1, j].imshow(y0[i, 0], aspect="auto", cmap="gray_r", origin="lower", interpolation="nearest")
    ax[1, j].set(title=f"its RFI mask ({frac[i]:.1%} of pixels)", xlabel="frequency channel", ylabel="time step")
plt.tight_layout(); plt.show()
rows = (y0[:, 0].mean(2) > 0.9).mean(); cols = (y0[:, 0].mean(1) > 0.9).mean()
print(f"frequency channels flagged at >90% of the time steps: {cols:.1%} | time steps flagged at >90% of the channels: {rows:.1%}")

**What to notice.** The RFI has a few clear shapes, and each shape is a different kind of transmitter:
**vertical lines** = one frequency channel that is on for a long time (a narrow-band station; some lines are dashed, i.e. the station switches on and off);
**horizontal lines** across all channels = a short broadband burst (an impulse, e.g. lightning); **short bursts confined to a block of channels** (here about channels 390–512);
and **isolated single pixels** scattered everywhere. Many masked pixels are invisible by eye in the image above them.
The two groups from the histograms are visible too: the bright waterfalls are smooth, the faint one (waterfall 27) shows wavy stripes from the sky signal.
So the network needs to see lines in **both** directions, and it must work at very different brightness levels.

### 0.4 The numbers that matter

In [ ]:
_, y_tr0 = load("train"); _, y_te0 = load("test")
print("split  waterfalls  RFI pixels")
for name, y in (("train", y_tr0), ("val", y0), ("test", y_te0)):
    print(f"{name:<6} {len(y):>9}   {y.mean():8.2%}")

**Classes:** about 2.75% of the pixels are RFI, the same in every split: a strong imbalance (36 clean pixels for every RFI pixel).
**Splits:** the Zenodo record ships 420 training and 140 test waterfalls; `test` is the published test set, unchanged. `val` is 60 of the 420 (fixed random choice), `train` is the other 360.
**Value range:** amplitudes from ~10⁻⁵ to ~400, always positive. There is no preprocessing: whatever you do to the input (a log, a normalisation per waterfall)
runs inside your 2 ms latency budget.

### 0.5 The metric, explained on the data
With only 2.75% RFI pixels, a model that flags nothing is 97% "accurate", so accuracy says nothing. **AUPRC** (the area under the precision–recall curve,
threshold-free) asks the real question: when you flag more and more pixels in order of your score, how many of the flagged pixels are really RFI (precision),
while you find more and more of the RFI (recall)? It ignores the millions of easy clean pixels and focuses on the rare class, as an observatory would:
missed RFI spoils the science, and over-flagging throws away sky data. F1 at a threshold chosen on `val` gives one operating point.

In [ ]:
del X0, y0, lx, m, z, y_tr0, y_te0

## 1. Data
One waterfall = 512 × 512 amplitudes; the mask marks RFI pixels (2.75% of them).

In [ ]:
from capstone.data.hera import load, threshold_flagger, auprc, best_f1, f1_at
X_tr, y_tr = load("train", max_items=40 if FAST else None)
X_va, y_va = load("val", max_items=10 if FAST else None)
X_te, y_te = load("test", max_items=20 if FAST else None)
print("train", X_tr.shape, "| val", X_va.shape, "| test", X_te.shape, "| RFI fraction (train):", round(float(y_tr.mean()), 4))

## 2. The classical baseline: a robust threshold
For every row, the median and the MAD of log|V|; a pixel's score is how many robust standard deviations it sits above the median.
The threshold is chosen on **val** (best F1), then applied to **test**.

In [ ]:
FLAG_THR = best_f1(y_va, threshold_flagger(X_va))[1]
s_te = threshold_flagger(X_te)
FLAG_AP, FLAG_F1 = auprc(y_te, s_te), f1_at(y_te, s_te, FLAG_THR)
print(f"threshold flagger, test: AUPRC {FLAG_AP:.4f}, F1 {FLAG_F1:.4f} (threshold {FLAG_THR:.2f}, chosen on val)")

## 3. A tiny U-Net, costed before training
The log and the normalisation are **part of the model** (`Prep`), so the timed call below includes them. One downsampling level, 8 channels.

In [ ]:
LX = np.log10(X_tr[:, :, ::8, ::8] + 1e-6)                    # normalisation constants from the training split
MU, SD = float(LX.mean()), float(LX.std())

class Prep(nn.Module):
    def __init__(self, mu, sd):
        super().__init__(); self.mu, self.sd = mu, sd
    def forward(self, x):
        return (torch.log10(x + 1e-6) - self.mu) / self.sd

def block(a, b, stride=1):
    return nn.Sequential(nn.Conv2d(a, b, 3, stride=stride, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())

class TinyUNet(nn.Module):
    def __init__(self, c=8):
        super().__init__()
        self.prep = Prep(MU, SD)
        self.enc, self.down = block(1, c), block(c, 2 * c, stride=2)
        self.up = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(2 * c, c, 3, padding=1), nn.ReLU())
        self.dec, self.out = block(2 * c, c), nn.Conv2d(c, 1, 1)
    def forward(self, x):
        e = self.enc(self.prep(x))
        return self.out(self.dec(torch.cat([e, self.up(self.down(e))], 1)))   # logits per pixel

model = TinyUNet()
p = profile(model, (1, 512, 512), bytes_per_weight=4, bytes_per_act=4)   # fp32 on the GPU
print(f"params {p['params']:,} | MACs per waterfall {p['macs'] / 1e9:.2f} G | peak activation (fp32) {p['peak_act_bytes'] / 1e6:.1f} MB")

## 4. Train on 128 × 128 tiles (the network is fully convolutional), evaluate on full waterfalls

In [ ]:
tile = lambda A: A.reshape(len(A), 1, 4, 128, 4, 128).transpose(0, 2, 4, 1, 3, 5).reshape(-1, 1, 128, 128)
T_X, T_y = tile(X_tr), tile(y_tr).astype(np.float32)
POS_W = torch.tensor(5.0, device=DEV)
fit(model, T_X, T_y, epochs=1 if FAST else 8, bs=32, lr=2e-3, device=DEV,
    loss_fn=lambda out, yb: F.binary_cross_entropy_with_logits(out, yb.float(), pos_weight=POS_W))
p_va, p_te = predict(model, X_va, bs=4, device=DEV), predict(model, X_te, bs=4, device=DEV)
THR = best_f1(y_va, p_va)[1]
NET_AP, NET_F1 = auprc(y_te, p_te), f1_at(y_te, p_te, THR)
print(f"tiny U-Net, test: AUPRC {NET_AP:.4f}, F1 {NET_F1:.4f}   (flagger: AUPRC {FLAG_AP:.4f}, F1 {FLAG_F1:.4f})")

## 5. Latency at batch 1, including the copies
`gpu_call` copies one waterfall (1 MB) to the GPU, runs the model and copies the scores back; `measure` repeats it and reports the median and the p99.

In [ ]:
x1 = torch.from_numpy(X_te[:1])
model = model.to(DEV).eval()
if DEV == "cuda":
    torch.cuda.reset_peak_memory_stats()
    LAT = measure(gpu_call(model, x1), cuda=True)
    FLAG_LAT = measure(lambda: threshold_flagger(x1.to("cuda")).cpu(), cuda=True)
    MEM_GB = torch.cuda.max_memory_allocated() / 1e9
    where = "T4 GPU, incl. copies"
else:
    torch.set_num_threads(1)
    LAT = measure(cpu_call(model, x1), n=50, warmup=5, repeats=3)
    FLAG_LAT = measure(lambda: threshold_flagger(x1), n=50, warmup=5, repeats=3)
    MEM_GB = float("nan")
    where = "CPU (1 thread): NOT the target; run on a T4 for the real number"
print(f"tiny U-Net: median {LAT['median_us'] / 1e3:.2f} ms, p99 {LAT['p99_us'] / 1e3:.2f} ms ({where}); peak GPU memory {MEM_GB:.3f} GB")
print(f"flagger:    median {FLAG_LAT['median_us'] / 1e3:.2f} ms, p99 {FLAG_LAT['p99_us'] / 1e3:.2f} ms")

In [ ]:
ROW = report("3 hera", (len(X_tr), len(X_va), len(X_te)), (1, 512, 512),
             f"U-Net AUPRC {NET_AP:.4f} F1 {NET_F1:.4f} | flagger AUPRC {FLAG_AP:.4f} F1 {FLAG_F1:.4f}", p,
             f"{where}: U-Net median {LAT['median_us'] / 1e3:.2f} / p99 {LAT['p99_us'] / 1e3:.2f} ms, flagger p99 {FLAG_LAT['p99_us'] / 1e3:.2f} ms; GPU mem {MEM_GB:.3f} GB",
             "CC BY 4.0 (Zenodo, confirmed)", extra=dict(minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.